# Batch Normalization va Layer Normalization voi tf.keras

**Muc tieu bai hoc:**
- Hieu **Batch Normalization (BatchNorm)** va **Layer Normalization (LayerNorm)** - hai ky thuat
  chuan hoa gia tri **trong luc huan luyen** (khac voi chuan hoa du lieu dau vao mot lan duy nhat
  truoc khi train), giup on dinh va tang toc qua trinh hoc.
- Phan biet ro **truc tinh thong ke** cua BatchNorm (theo tung vi du trong batch) va LayerNorm
  (theo tung feature trong 1 vi du) - diem khac biet cot loi giua hai ky thuat.
- Biet chen `layers.BatchNormalization()`/`layers.LayerNormalization()` dung vi tri trong kien
  truc `tf.keras`, va quan sat hieu qua thuc te cua chung khi mang bi khoi tao kem.

In [ ]:
# --- Google Colab setup: bo qua tu dong neu chay Jupyter local ---
import sys, os

os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"  # tat toi uu oneDNN de dam bao ket qua giong het nhau
# giua cac lan chay - phai dat TRUOC khi 'import tensorflow' o cell sau (oneDNN co the doi thu
# tu tinh toan tren nhieu luong CPU, gay lech ket qua nho o vai chu so thap phan cuoi giua cac lan)

if "google.colab" in sys.modules:
    REPO_DIR = "/content/AI-Teaching-Labs"
    if not os.path.isdir(REPO_DIR):
        !git clone --depth 1 https://github.com/NonomiyaIzumi/AI-Teaching-Labs.git {REPO_DIR}
    NOTEBOOK_DIR = f"{REPO_DIR}/Deep Learning/Module 1/keras-tensorflow/05_Batch-Norm-Layer-Norm"
    os.chdir(NOTEBOOK_DIR)
    print("Da chuyen working directory sang:", os.getcwd())
else:
    print("Khong phai Colab - dung working directory hien tai:", os.getcwd())
    print("Neu chay Jupyter local: tu thu muc keras-tensorflow/, chay 'uv sync' truoc.")

## 1. Vi sao can chuan hoa NGAY TRONG luc train?

Bai truoc da thay: khoi tao trong so kem co the khien phan phoi cua `Z^[l] = W^{[l]}A^{[l-1]}+b^{[l]}`
"troi" ve vung qua lon hoac qua nho ngay tu dau, gay vanishing/exploding gradient. Nhung ngay ca voi
khoi tao tot, phan phoi cua `Z^[l]` van co the **thay doi dan** trong qua trinh train khi trong so
cac lop truoc duoc cap nhat - hien tuong nay goi la **internal covariate shift**: moi lop phai lien
tuc "thich nghi" voi phan phoi dau vao dang thay doi cua no.

**Y tuong cua BatchNorm/LayerNorm:** thay vi chi chon khoi tao that can than **mot lan** (bai
truoc), ta **chu dong chuan hoa lai** `Z^[l]` (tru trung binh, chia do lech chuan) **o moi buoc
forward**, giu phan phoi on dinh xuyen suot qua trinh train - khong chi luc khoi tao. Sau khi chuan
hoa ve trung binh 0/phuong sai 1, mot cap tham so hoc duoc `\gamma, \beta` (scale & shift) cho phep
mang **tu quyet dinh** neu no van can mot phan phoi khac 0/1 o buoc do:

$$\hat{Z} = \frac{Z - \mu}{\sqrt{\sigma^2 + \varepsilon}} \qquad\qquad \tilde{Z} = \gamma \hat{Z} + \beta$$

## 2. Batch Normalization vs Layer Normalization

Khac biet duy nhat giua hai ky thuat la **truc tinh** `\mu, \sigma^2`:

| | **BatchNormalization** | **LayerNormalization** |
|---|---|---|
| Tinh thong ke tren | Tung **feature**, gop theo **cac vi du trong batch** | Tung **vi du**, gop theo **cac feature** |
| Phu thuoc batch size? | **Co** - batch qua nho lam thong ke khong on dinh | **Khong** - tinh doc lap tren tung vi du |
| Train vs Test | Can co che rieng: dung thong ke cua batch khi train, dung trung binh truot (`moving_mean`/`moving_variance`) khi test | Giong het nhau o train va test (khong phu thuoc batch) |
| Thuong dung cho | CNN/MLP voi batch size du lon | RNN/Transformer, hoac khi batch size nho/thay doi |

BatchNorm can co che "train/test khac nhau" vi luc test co the chi co **1** vi du - khong the tinh
`\mu, \sigma^2` tren batch. Giai phap: trong luc train, ngoai chuan hoa theo batch hien tai, mang
con duy tri mot **trung binh truot** (exponential moving average) cua `\mu, \sigma^2` qua cac batch;
luc test (`training=False`), dung truc tiep trung binh truot nay thay vi tinh lai tren batch (co
the chi co 1 vi du).

## 3. Du lieu & kien truc thi nghiem

De thay ro hieu qua "cuu" mang cua BatchNorm/LayerNorm, ta co tinh dung mot kien truc **sau**
(`[8, 20, 20, 20, 20, 1]`, 4 hidden layer) **voi khoi tao trong so kem** (`RandomNormal(stddev=4.0)`
- phuong sai lon hon nhieu so muc an toan) tren bo du lieu y te that **Pima Indians Diabetes** (768
benh nhan, 8 dac trung lam sang, du doan tieu duong) - dung dieu kien de vanishing/exploding
gradient the hien ro, roi quan sat BatchNorm/LayerNorm co giup mang hoc duoc khong.

In [ ]:
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from keras import layers

from keras_utils import load_dataset, compile_and_train, evaluate_classification

tf.config.experimental.enable_op_determinism()  # dam bao ket qua giong het nhau giua cac lan chay

train_X, train_Y, test_X, test_Y = load_dataset()
print("train_X:", train_X.shape, " train_Y:", train_Y.shape)

## 4. Chen lop chuan hoa vao kien truc - Bai tap

Vi tri chen chuan: **giua** phep bien doi tuyen tinh (`Dense`, KHONG activation) **va** ham kich
hoat phi tuyen (`Activation("relu")`) - chuan hoa truoc, roi moi ap phi tuyen len gia tri da chuan
hoa. Day la ly do lop `Dense` duoi day khong khai bao `activation="relu"` truc tiep, ma tach rieng
thanh `layers.Activation("relu")` o sau lop chuan hoa.

### Bai tap - `build_model_with_norm`

In [ ]:
def build_model_with_norm(input_dim, norm_type="none", bad_init_stddev=4.0):
    '''
    Xay dung mang [input_dim, 20, 20, 20, 20, 1] voi khoi tao trong so CO CHU Y kem
    (RandomNormal stddev=4.0), chen them lop chuan hoa (neu co) GIUA moi Dense va ReLU
    tuy `norm_type`.

    Arguments:
    norm_type -- "none" | "batchnorm" | "layernorm"

    Returns: mot tf.keras.Model CHUA compile.
    '''
    bad_init = keras.initializers.RandomNormal(mean=0.0, stddev=bad_init_stddev, seed=3)
    layer_list = [keras.Input(shape=(input_dim,))]
    for _ in range(4):
        layer_list.append(layers.Dense(20, kernel_initializer=bad_init))
        # (~4 dong) neu norm_type == "batchnorm": them layers.BatchNormalization()
        #           neu norm_type == "layernorm": them layers.LayerNormalization()
        #           neu norm_type == "none":      khong them gi
        #           sau do LUON them layers.Activation("relu")
        # YOUR CODE STARTS HERE
        pass
        # YOUR CODE ENDS HERE
    layer_list.append(layers.Dense(1, kernel_initializer=bad_init, activation="sigmoid"))
    model = keras.Sequential(layer_list)
    return model

In [ ]:
#@title Answer { display-mode: "form" }
def build_model_with_norm(input_dim, norm_type="none", bad_init_stddev=4.0):
    '''
    Xay dung mang [input_dim, 20, 20, 20, 20, 1] voi khoi tao trong so CO CHU Y kem
    (RandomNormal stddev=4.0), chen them lop chuan hoa (neu co) GIUA moi Dense va ReLU
    tuy `norm_type`.

    Arguments:
    norm_type -- "none" | "batchnorm" | "layernorm"

    Returns: mot tf.keras.Model CHUA compile.
    '''
    bad_init = keras.initializers.RandomNormal(mean=0.0, stddev=bad_init_stddev, seed=3)
    layer_list = [keras.Input(shape=(input_dim,))]
    for _ in range(4):
        layer_list.append(layers.Dense(20, kernel_initializer=bad_init))
        # (~4 dong)
        # YOUR CODE STARTS HERE
        if norm_type == "batchnorm":
            layer_list.append(layers.BatchNormalization())
        elif norm_type == "layernorm":
            layer_list.append(layers.LayerNormalization())
        layer_list.append(layers.Activation("relu"))
        # YOUR CODE ENDS HERE
    layer_list.append(layers.Dense(1, kernel_initializer=bad_init, activation="sigmoid"))
    model = keras.Sequential(layer_list)
    return model

## 5. Huan luyen & so sanh none / batchnorm / layernorm

Dung `learning_rate=0.1`, full-batch gradient descent (`batch_size=None`) cho ca 3 bien the.

In [ ]:
EPOCHS = 2000
histories = {}
for norm_type in ["none", "batchnorm", "layernorm"]:
    tf.random.set_seed(3)
    model = build_model_with_norm(train_X.shape[0], norm_type=norm_type)
    history = compile_and_train(
        model, train_X, train_Y,
        optimizer=keras.optimizers.SGD(learning_rate=0.1),
        epochs=EPOCHS, batch_size=None, verbose=0,
    )
    histories[norm_type] = {"model": model, "history": history}
    print(f'{norm_type:10s} -> cost cuoi cung (train) = {history.history["loss"][-1]:.4f}')

In [ ]:
plt.figure(figsize=(7, 4.5))
for norm_type in ["none", "batchnorm", "layernorm"]:
    plt.plot(histories[norm_type]["history"].history["loss"], label=norm_type)
plt.xlabel("epoch")
plt.ylabel("cost")
plt.title("Cost qua qua trinh train - so sanh none / batchnorm / layernorm")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
for norm_type in ["none", "batchnorm", "layernorm"]:
    evaluate_classification(histories[norm_type]["model"], test_X, test_Y, f'norm_type = "{norm_type}" (test set)')

## 6. Ket qua & Phan tich

Sau 2000 epoch (`learning_rate=0.1`): **`none`** cost dung yen o **~0.647** (~`ln(2)`, dung bang
cost cua mot mo hinh chi doan lop da so), test Accuracy ~65% nhung **Precision = Recall = 0** -
mang hoan toan khong hoc duoc gi voi khoi tao kem tren kien truc sau nay. **`batchnorm`** va
**`layernorm`** deu "cuu" duoc mang: cost train giam manh ve gan 0, test Accuracy len **~65-70%**
cho ca hai (Recall tu ~45% den ~60% tuy lan chay - xem luu y ben duoi).

**Luu y ve reproducibility:** voi `batchnorm`/`layernorm`, cac con so cu the co the lech nhau vai
phan tram giua cac lan chay du da co dinh seed - mang cang sau va cang nhieu epoch thi phep tinh
gradient descent tren CPU nhieu luong cang co co hoi tich luy sai so lam tron nho, du ban chat dinh
tinh (`none` that bai hoan toan, `batchnorm`/`layernorm` cuu duoc mang) luon nhat quan.

**Luu y ve overfitting:** cost train gan 0 trong khi test accuracy chi ~65-70% la dau hieu ro ret
cua **overfitting** - voi 4 hidden layer x 20 unit tren chi 614 vi du train, mang du suc "hoc
thuoc" toan bo tap train sau 2000 epoch full-batch. Day la mot diem quan trong can luu y: BatchNorm/
LayerNorm giai quyet van de vanishing/exploding gradient, nhung **khong** tu dong chong overfitting
- neu muon tranh ca hai van de cung luc, can ket hop them ky thuat khac (regularization, dropout,
early stopping...).

**Diem mau chot:** cung mot kien truc, cung mot khoi tao kem, chi them mot lop chuan hoa dung vi tri
da bien mot mang "chet" (Recall = 0) thanh mot mang hoc duoc tin hieu that su. `layers.
BatchNormalization()` tu dong phan biet `training=True` (dung thong ke cua batch hien tai) va
`training=False` (dung trung binh truot) - Keras xu ly hoan toan viec chuyen doi nay o phia sau,
ban khong can tu quan ly 2 nhanh logic train/test.

## 7. Bai tap mo rong

1. **Batch size = 1:** thu train `batchnorm` voi `batch_size=1` (giu nguyen so epoch tuong duong)
   - du doan dieu gi se xay ra voi thong ke `\mu, \sigma^2` tinh tren batch chi co 1 diem? So sanh
   voi `layernorm` trong cung tinh huong (LayerNorm khong phu thuoc batch size).
2. **Chuan hoa ca lop output:** thu them `BatchNormalization`/`LayerNormalization` ngay truoc lop
   `Dense(1, activation="sigmoid")` cuoi cung (thuc hanh chuan thuong KHONG chuan hoa lop output) -
   quan sat cost/accuracy co thay doi khong, giai thich tai sao.
3. **Giam kien truc:** thu kien truc nong hon (vd chi 1-2 hidden layer thay vi 4) voi cung khoi tao
   kem - `none` co con that bai hoan toan khong? Dieu nay cho thay muc do nghiem trong cua vanishing/
   exploding gradient phu thuoc the nao vao do sau mang.
4. **So sanh voi khoi tao tot:** thay `bad_init_stddev=4.0` bang `keras.initializers.HeNormal()`
   (khoi tao chuan cho ReLU) cho ca 3 bien the - BatchNorm/LayerNorm co con tao ra khac biet lon
   nhu vay khi khoi tao da tot san tu dau khong?

## 8. Tai lieu tham khao

| Chu de | Lien ket |
|---|---|
| Ioffe & Szegedy, 2015 - Batch Normalization | https://arxiv.org/abs/1502.03167 |
| Ba, Kiros & Hinton, 2016 - Layer Normalization | https://arxiv.org/abs/1607.06450 |
| `tf.keras.layers.BatchNormalization` API | https://www.tensorflow.org/api_docs/python/tf/keras/layers/BatchNormalization |
| `tf.keras.layers.LayerNormalization` API | https://www.tensorflow.org/api_docs/python/tf/keras/layers/LayerNormalization |
| Pima Indians Diabetes Dataset (nguon goc) | https://www.kaggle.com/datasets/uciml/pima-indians-diabetes-database |